# Corrigé : Le projet fil rouge (chapitre 1)

*MLSMM2151 Data Science · Séance 1 · Corentin Vande Kerckhove*

Corrigé des quatre hack-times du chapitre 1. Les solutions sont
commentées : ce qui compte n'est pas le code, mais le raisonnement.

## Hack-time 1 : règle à la main contre règle apprise

**1. Un studio.** La règle à la main donne 30 euros, la règle apprise 60. La
seconde est plus crédible : un logement, même pour une personne, a des coûts
qui ne dépendent pas du nombre d'occupants (ménage, draps, déplacement), et les
huit exemples le montrent, puisque le logement d'une personne y coûte 60 euros.
La règle à la main l'ignore parce que son auteur pensait aux logements
familiaux. C'est le premier apport d'une règle apprise : elle **révèle une
structure que l'intuition avait écrasée**.

**2. Vingt personnes.** 600 euros contre 470. Aucun des huit exemples ne dépasse
huit personnes : les deux règles **extrapolent**, et rien dans les données ne
permet de départager. La règle apprise donne pourtant son chiffre avec le même
aplomb que pour cinq personnes. C'est son défaut caractéristique : elle ne
sait pas qu'elle ne sait pas. On y reviendra au chapitre 5.

**3. Ce qui manque.** Du volume (huit exemples, c'est huit), de la variété (que
des logements de 1 à 8 personnes), et d'autres variables (la commune, le type
de logement, les équipements, les avis). Et une façon de **vérifier** la règle
sur des logements qu'elle n'a pas vus, plutôt que de la croire sur parole.
C'est exactement le programme du cours.

### Le défi optionnel : un exemple aberrant

Avec un logement de 2 personnes à 400 euros ajouté aux huit autres, la
prédiction pour cinq personnes passe d'environ 146 à environ 164 euros : une
seule ligne sur neuf a déplacé le résultat de 12 %. Mais le plus frappant est
ailleurs : la règle a **changé de nature**. Le coût fixe passe d'environ 39 à
environ 118 euros, et la part par personne de 22 à 9 euros. La droite s'est
couchée pour se rapprocher du point aberrant, et elle ne décrit plus du tout
les huit autres logements. La règle à la main, elle, n'a pas bougé. Ce n'est ni une qualité ni un défaut
en soi : c'est une **sensibilité aux données** qu'il faut connaître. Elle fait
qu'une règle apprise s'améliore quand les données s'améliorent, et qu'elle se
dégrade quand une valeur aberrante s'y glisse sans qu'on la voie. D'où le
chapitre 3, entièrement consacré à regarder les données avant de les utiliser.

## Hack-time 2 : six demandes, quatre paradigmes

| # | Demande | Paradigme | Pourquoi |
|---|---|---|---|
| 1 | Durée de traitement d'une commande | **Supervisé, régression** | On prédit un nombre, et on a des durées passées connues |
| 2 | Regrouper 40 000 clients | **Non supervisé** | Aucune réponse connue, on cherche une structure |
| 3 | Fraude, avec 200 000 cas étiquetés | **Supervisé, classification** | Deux catégories, et on sait lesquelles |
| 4 | Prix d'un billet, heure par heure | **Renforcement** | Décisions successives, récompense différée, et chaque décision modifie l'état du système |
| 5 | 50 transactions les plus atypiques | **Non supervisé** | Aucune fraude connue : on cherche ce qui s'écarte de la norme |
| 6 | Dépense de l'an prochain | **Supervisé, régression** | Un montant à prédire, à partir d'historiques |

**Les deux qui se ressemblent : 3 et 5.** Les deux portent sur des transactions
suspectes. Ce qui les sépare n'est ni le métier ni les données, c'est
**l'existence d'étiquettes**.

Dans le cas 3, quelqu'un a marqué 200 000 transactions comme frauduleuses ou
non. On peut apprendre à reproduire ce jugement, et on peut mesurer si on y
arrive.

Dans le cas 5, personne n'a rien marqué. On ne peut que détecter ce qui est
inhabituel. Une transaction atypique n'est pas forcément une fraude : elle peut
être un gros achat parfaitement légitime. Et surtout, **on ne peut pas mesurer
si on a raison**.

**Si l'entreprise commence à étiqueter ses fraudes**, le problème bascule du non
supervisé vers le supervisé. C'est le chemin habituel : on démarre en détection
d'anomalies, les analystes confirment ou infirment les alertes, et ces
confirmations constituent progressivement le jeu étiqueté qui permettra
d'entraîner un modèle supervisé.

Une remarque qui vaut d'être faite en séance : le paradigme n'est pas une
propriété du problème métier, c'est une propriété de **ce dont on dispose**.

## Hack-time 3 : diagnostic des manquants

In [ ]:
import pandas as pd

BASE = "https://github.com/cvandekerckh/mlsmm2151/releases/download/v1-donnees"
annonces = pd.read_csv(f"{BASE}/airbnb_bruxelles_raw.csv", low_memory=False)

# 1. Taux de manquants, du plus au moins manquant
manquants = annonces.isna().mean().sort_values(ascending=False)
print(manquants.head(12).round(3).to_string())

In [ ]:
# 2. La colonne sans aucune information
vides = manquants[manquants == 1.0]
print("Colonnes entièrement vides :", list(vides.index))

**`license` est vide à 100 %.** Aucune annonce bruxelloise n'a de numéro de
licence renseigné dans ce snapshot. La colonne existe dans le format d'Inside
Airbnb, mais elle n'est pas alimentée ici.

Une colonne constante, ou entièrement vide, n'apporte strictement rien à un
modèle : elle ne permet de distinguer aucune ligne d'une autre. C'est le premier
type de colonne à retirer, et c'est aussi le plus facile à rater si on ne
regarde jamais les taux de manquants.

**3. Que fait-on d'une colonne vide ?** On la retire : c'est la seule décision
sans regret de tout le nettoyage. Mais pas maintenant. Ici, nous constatons,
nous ne modifions rien ; le traitement des manquants est le chapitre 3.

La distinction n'est pas byzantine : nettoyer au fur et à mesure qu'on découvre
est la meilleure façon de prendre des décisions qu'on ne saura plus justifier
deux semaines plus tard. Et une colonne vide est facile à rater précisément
parce qu'elle ne fait pas d'erreur : elle ne gêne rien, elle n'apporte rien.

### Le défi optionnel : pourquoi cinq colonnes manquent-elles ensemble ?

In [ ]:
colonnes_avis = ["first_review", "last_review", "review_scores_rating",
                 "review_scores_cleanliness", "reviews_per_month"]

# Ces colonnes manquent-elles sur exactement les mêmes lignes ?
manque_ensemble = annonces[colonnes_avis].isna().all(axis=1)
manque_au_moins_une = annonces[colonnes_avis].isna().any(axis=1)
print(f"Lignes où les 5 manquent   : {manque_ensemble.sum()}")
print(f"Lignes où au moins 1 manque : {manque_au_moins_une.sum()}")

In [ ]:
# L'explication tient en une ligne
print(annonces.loc[manque_ensemble, "number_of_reviews"].value_counts().head())

**Les 925 lignes concernées ont exactement zéro avis.** Ce ne sont pas des
données perdues : ce sont des annonces qui n'ont jamais été commentées. Il n'y a
rien à récupérer, parce qu'il n'y a jamais rien eu.

Trois conséquences, et c'est là tout l'intérêt de la question.

**Imputer par la moyenne serait une faute.** Attribuer la note moyenne, 4,7, à
une annonce sans aucun avis revient à inventer une réputation qui n'existe pas,
et à la donner précisément aux annonces les plus incertaines.

**L'absence est elle-même une information.** Une annonce sans avis est
probablement récente, ou peu réservée. C'est prédictif. La bonne pratique est de
créer une variable `a_des_avis`, qui conserve cette information au lieu de la
noyer dans une imputation.

**Ces manquants ne sont pas aléatoires.** On parle de manquants *MNAR*, dont
l'absence dépend de la valeur elle-même. C'est le cas le plus délicat, et le
chapitre 3 y revient.

## Hack-time 4 : prix médian par type de logement

In [ ]:
train = pd.read_csv(f"{BASE}/airbnb_bruxelles_train.csv")

par_type = (
    train.groupby("room_type")["y_price"]
    .agg(["median", "count"])
    .sort_values("count", ascending=False)
    .round(1)
)
par_type

**Ce qu'on répond à la responsable du revenue management.**

Une réponse acceptable : *« Le prix médian d'une chambre privée est d'environ
60 euros, sur un peu plus de mille annonces. C'est un point de départ solide,
mais c'est une médiane globale : elle ne tient compte ni de la commune, ni de la
capacité, ni de la saison. Pour un prix par annonce, il faut un modèle. »*

Ce qui est attendu ici n'est pas le chiffre, c'est le **réflexe d'accompagner la
médiane de son effectif et de ses limites**. Une réponse qui donne un prix sans
préciser sur combien d'annonces il repose est une réponse incomplète.

Notez au passage la justification du cours entier : une médiane par catégorie est
un modèle, très simple. Toute la suite consiste à faire mieux, et à savoir
mesurer de combien.

### Le défi optionnel : le prix médian par commune

In [ ]:
par_commune = (
    train.groupby("neighbourhood_cleansed")["y_price"]
    .agg(["median", "count"])
    .sort_values("median", ascending=False)
    .round(1)
)
par_commune.head(8)

La commune la plus chère est **Bruxelles**, à 99 euros de médiane. Et cette fois,
avec 1 279 annonces, le chiffre est solide : le centre-ville est réellement le
plus cher, ce qui n'a rien de surprenant.

**Le résultat n'est donc pas un artefact.** C'est important de le dire, parce que
la vérification n'a pas pour but de trouver un piège à tous les coups : elle a
pour but de savoir si on peut faire confiance au chiffre. Ici, oui.

Regardez tout de même les rangs suivants. **Woluwe-Saint-Pierre est troisième
avec 70 annonces, Berchem-Sainte-Agathe cinquième avec 30**, au milieu de
communes qui en comptent des centaines. Ces deux positions-là ne reposent pas sur
grand-chose. Filtrons pour voir.

In [ ]:
fiables = par_commune[par_commune["count"] >= 100].sort_values("median", ascending=False)
fiables.head(5)

Le haut du classement ne bouge pas, mais les deux petites communes disparaissent
et le classement devient entièrement interprétable.

**La leçon, qui vaut pour tout le cours.** Un tri par moyenne ou par médiane
remonte mécaniquement les groupes à faible effectif, dans les deux sens : ils
sont surreprésentés aux deux extrémités du classement, simplement parce qu'ils
varient plus. Le filtre sur l'effectif ne change pas toujours la conclusion, et
c'est très bien : ce qui compte est que vous sachiez, après l'avoir posé, si
votre classement décrit un phénomène ou du bruit.

Un mot sur `Shared room` et `Hotel room` dans le tableau précédent : **13 et
8 annonces**. Leurs médianes, 25 et 126 euros, ne devraient jamais être citées
sans leur effectif. C'est un piège qui reviendra tout au long du cours.